## Advanced amazon delivery predictions

In [1]:
import numpy as np
import pandas as pd
# To use IterativeImputer (MICE) in scikit-learn, we explicitly enable it first
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

df = pd.read_csv("data/amazon_delivery.csv")

print(df.shape)


(43739, 16)


In [2]:
# Audit data
print(df.isna().sum())

Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather            91
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64


In [3]:
# Pre-Imputation Sanitization (Sanitizing the rating anomaly and driver age minimums)
df['Agent_Rating'] = df['Agent_Rating'].clip(lower=1.0, upper=5.0)
df['Agent_Age'] = df['Agent_Age'].clip(lower=18)

print(df.shape)

(43739, 16)


In [5]:

# Create discrete integer codes for the categorical Weather feature so MICE can compute it
# Map missing values cleanly by safely handling the dropped null representation
weather_mapping = {val: idx for idx, val in enumerate(df['Weather'].dropna().unique())}
inverse_weather_mapping = {idx: val for val, idx in weather_mapping.items()}
df['Weather_Encoded'] = df['Weather'].map(weather_mapping)

# Track temporal cycles to serve as a predictive reference feature for MICE
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Order_Month'] = df['Order_Date'].dt.month

# Isolate internal correlated reference metrics
mice_features = ['Agent_Age', 'Agent_Rating', 'Delivery_Time', 'Order_Month', 'Weather_Encoded']
mice_df = df[mice_features].copy()

# Initialize and run MICE ==> Unconstrained inside sklearn to avoid indexing ValueError
mice_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10, 
    random_state=42
)

imputed_array = mice_imputer.fit_transform(mice_df)
imputed_df = pd.DataFrame(imputed_array, columns=mice_features, index=df.index)

### Instead of configuring constraints inside the imputer, clean the output DataFrame explicitly

# Clip imputed driver ratings safely back to 1.0 - 5.0 range
imputed_df['Agent_Rating'] = imputed_df['Agent_Rating'].clip(lower=1.0, upper=5.0)

# Clip the encoded weather variables to their exact index bounds, then round to the nearest code
max_weather_idx = len(weather_mapping) - 1
imputed_df['Weather_Encoded'] = imputed_df['Weather_Encoded'].clip(lower=0, upper=max_weather_idx)
imputed_df['Weather_Encoded'] = np.round(imputed_df['Weather_Encoded']).astype(int)

# 6. Map the complete fields back into the main DataFrame
df['Agent_Rating'] = imputed_df['Agent_Rating']
df['Weather'] = imputed_df['Weather_Encoded'].map(inverse_weather_mapping)

# Drop tracking placeholder proxies
df = df.drop(columns=['Weather_Encoded', 'Order_Month'])

print("\nMissing values check following corrected MICE execution:")
print(df[['Weather', 'Agent_Rating']].isna().sum())


Missing values check following corrected MICE execution:
Weather         0
Agent_Rating    0
dtype: int64
